# 03 — Carbon Stock Analysis

Estimate above-ground biomass (Mg/ha) and carbon stock (tCO2e/ha) from spectral indices using `climatevision.models.regression.BiomassRegressor`.

**Pipeline**

1. Load (or simulate) a labelled dataset of spectral indices ↔ biomass.
2. Train a Random Forest regressor and evaluate on a held-out split.
3. Convert biomass predictions to carbon and CO₂e using IPCC defaults.
4. Inspect feature importances to confirm the model is leaning on the indices we expect (NDVI, EVI, NIR).
5. Persist the trained regressor + metrics so the analytics API can serve them.

## Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from climatevision.models.regression import (
    BiomassRegressor,
    biomass_to_carbon,
    biomass_to_co2e,
    evaluate_regression,
    serialize_metrics,
)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUTS = PROJECT_ROOT / "outputs" / "carbon"
OUTPUTS.mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(42)

## 1. Load training data

If a real labelled dataset is available at `data/biomass/biomass_samples.parquet`, load it. Otherwise simulate a plausible one so the notebook is runnable in CI.

In [ ]:
DATA_PATH = PROJECT_ROOT / "data" / "biomass" / "biomass_samples.parquet"
FEATURE_COLS = ["ndvi", "evi", "savi", "ndmi", "nbr", "red", "green", "blue", "nir", "swir1"]

if DATA_PATH.exists():
    df = pd.read_parquet(DATA_PATH)
    print(f"Loaded {len(df):,} real samples from {DATA_PATH}")
else:
    n = 5_000
    X = rng.uniform(0, 1, size=(n, len(FEATURE_COLS)))
    biomass = (
        220 * X[:, 0]   # NDVI
        + 80 * X[:, 1]  # EVI
        + 30 * X[:, 8]  # NIR
        - 20 * X[:, 5]  # Red
        + rng.normal(0, 8, size=n)
    )
    df = pd.DataFrame(X, columns=FEATURE_COLS)
    df["biomass_mg_ha"] = np.clip(biomass, 0, None)
    print(f"No real dataset found, simulated {n:,} samples")
df.head()

## 2. Train / test split

In [ ]:
split_idx = int(0.8 * len(df))
perm = rng.permutation(len(df))
train_idx, test_idx = perm[:split_idx], perm[split_idx:]

X_train = df.loc[train_idx, FEATURE_COLS].to_numpy()
y_train = df.loc[train_idx, "biomass_mg_ha"].to_numpy()
X_test = df.loc[test_idx, FEATURE_COLS].to_numpy()
y_test = df.loc[test_idx, "biomass_mg_ha"].to_numpy()

print(f"train={X_train.shape[0]:,}  test={X_test.shape[0]:,}")

## 3. Train a Random Forest regressor

In [ ]:
regressor = BiomassRegressor(
    model_type="random_forest",
    feature_names=FEATURE_COLS,
    model_kwargs={"n_estimators": 300, "min_samples_leaf": 2},
)
regressor.fit(X_train, y_train)

metrics = regressor.evaluate(X_test, y_test)
print(f"RMSE = {metrics.rmse:.2f} Mg/ha")
print(f"MAE  = {metrics.mae:.2f} Mg/ha")
print(f"R^2  = {metrics.r2:.3f}")
print(f"MAPE = {metrics.mape:.2%}")

## 4. Convert to carbon and CO₂e

In [ ]:
predicted_biomass = regressor.predict(X_test)
predicted_carbon = biomass_to_carbon(predicted_biomass)
predicted_co2e = biomass_to_co2e(predicted_biomass)

summary = pd.DataFrame({
    "biomass_mg_ha": predicted_biomass,
    "carbon_t_ha": predicted_carbon,
    "co2e_t_ha": predicted_co2e,
})
summary.describe().round(2)

## 5. Feature importances

In [ ]:
importances = regressor.feature_importances()
imp_df = pd.Series(importances).sort_values(ascending=False)
imp_df

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
imp_df.plot.bar(ax=ax)
ax.set_title("Feature importances — biomass regressor")
ax.set_ylabel("Importance")
plt.tight_layout()
fig.savefig(OUTPUTS / "feature_importances.png", dpi=150)
plt.close(fig)
print(f"Wrote {OUTPUTS / 'feature_importances.png'}")

## 6. Persist artifacts

In [ ]:
model_path = regressor.save(PROJECT_ROOT / "models_pretrained" / "biomass_rf.pkl")
metrics_path = serialize_metrics(metrics, OUTPUTS / "metrics.json")
print(f"Model:   {model_path}")
print(f"Metrics: {metrics_path}")

## Next steps

- See `04_model_validation.ipynb` for a held-out validation sweep across the Amazon, Congo, and Southeast Asia regions.
- See `05_impact_reporting.ipynb` for how to plug these carbon estimates into a stakeholder report.